# Gold Layer — IT Support Tickets

In this notebook, I use the cleaned Silver data to create a few tables for reporting in Power BI. I want to look at ticket volume, resolution performance and customer satisfaction.

## 1. Load the Silver data

I start by reading the Silver table and selecting the columns I need for the reports.

In [0]:
from pyspark.sql import functions as F

silver = spark.table("db_projects.data_engineering.ID001_Silver_Tickets")

report_data = silver.select(
    "ticket_id", "created_at", "priority", "status", "issue_type",
    "region_name", "is_resolved", "resolution_time_hours", "csat_valid"
)

display(report_data.limit(5))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7466101894036780>, line 10
      3 silver = spark.table("db_projects.data_engineering.ID001_Silver_Tickets")
      5 report_data = silver.select(
      6     "ticket_id", "created_at", "priority", "status", "issue_type",
      7     "region_name", "is_resolved", "resolution_time_hours", "csat_valid"
      8 )
---> 10 display(report_data.limit(5))

File <command-7466101894036780>, line 5
      1 from pyspark.sql import functions as F
      3 silver = spark.table("db_projects.data_engineering.ID001_Silver_Tickets")
----> 5 report_data = silver.select(
      6     "ticket_id", "created_at", "priority", "status", "issue_type",
      7     "region_name", "is_resolved", "resolution_time_hours", "csat_valid"
      8 )
     10 display(report_data.limit(5))

File <command-7466101894036780>, line 3
      1 from pyspark.sql import fu

## 2. Overall ticket KPIs

I calculate the main numbers for the dashboard. For average resolution time, I only include tickets marked as resolved. CSAT uses the valid scores from Silver.

In [0]:
gold_overview = report_data.agg(
    F.count("*").alias("total_tickets"),
    F.sum(F.col("is_resolved").cast("int")).alias("resolved_tickets"),
    F.round(F.avg(F.col("is_resolved").cast("double")) * 100, 2).alias("resolution_rate_pct"),
    F.round(F.avg(F.when(F.col("is_resolved"), F.col("resolution_time_hours"))), 2).alias("avg_resolution_hours"),
    F.round(F.avg("csat_valid"), 2).alias("avg_csat")
)

display(gold_overview)

## 3. Monthly trends

I group tickets by month to see how the volume and resolution rate change over time.

In [0]:
gold_monthly = (
    report_data
    .filter(F.col("created_at").isNotNull())
    .withColumn("month", F.date_trunc("month", "created_at").cast("date"))
    .groupBy("month")
    .agg(
        F.count("*").alias("total_tickets"),
        F.sum(F.col("is_resolved").cast("int")).alias("resolved_tickets"),
        F.round(F.avg(F.col("is_resolved").cast("double")) * 100, 2).alias("resolution_rate_pct")
    )
    .orderBy("month")
)

display(gold_monthly)

## 4. Performance by priority

I compare how many tickets are in each priority level and how long resolved tickets take to close.

In [0]:
gold_priority = (
    report_data
    .groupBy("priority")
    .agg(
        F.count("*").alias("total_tickets"),
        F.sum(F.col("is_resolved").cast("int")).alias("resolved_tickets"),
        F.round(F.avg(F.col("is_resolved").cast("double")) * 100, 2).alias("resolution_rate_pct"),
        F.round(F.avg(F.when(F.col("is_resolved"), F.col("resolution_time_hours"))), 2).alias("avg_resolution_hours")
    )
    .orderBy("priority")
)

display(gold_priority)

## 5. Performance by region

I also compare ticket volumes, resolution rates and customer satisfaction between regions.

In [0]:
gold_region = (
    report_data
    .groupBy("region_name")
    .agg(
        F.count("*").alias("total_tickets"),
        F.sum(F.col("is_resolved").cast("int")).alias("resolved_tickets"),
        F.round(F.avg(F.col("is_resolved").cast("double")) * 100, 2).alias("resolution_rate_pct"),
        F.round(F.avg("csat_valid"), 2).alias("avg_csat")
    )
    .orderBy("region_name")
)

display(gold_region)

## 6. Save the Gold tables

Finally, I save the four reporting tables in Delta format. I'll use them later when building the Power BI dashboard.

In [0]:
gold_tables = {
    "ID001_Gold_Overview": gold_overview,
    "ID001_Gold_Monthly": gold_monthly,
    "ID001_Gold_Priority": gold_priority,
    "ID001_Gold_Region": gold_region
}

for table_name, dataframe in gold_tables.items():
    full_name = f"db_projects.data_engineering.{table_name}"
    dataframe.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"Saved {full_name}: {dataframe.count()} rows")